# MM-Net — external validation on ISRUC-Sleep, both heads

The Sleep-EDF experiment could only test the staging head, because that corpus has no
cardiorespiratory channels and no respiratory-event annotations. ISRUC-Sleep-II does. It
supplies the four iSLEEPS EEG derivations *verbatim* (C4, C3, O2, O1 against the
contralateral mastoid), two EOG channels, chin EMG, ECG, airflow, thoracic and abdominal
effort, SpO2 — and scored obstructive, central and mixed apneas and hypopneas.

So this is the first external test of the **respiratory** head, and a staging test with
**no montage confound** — the derivation mismatch that cost N3 recall on Sleep-EDF does
not apply here.

**Protocol.** Identical to the Sleep-EDF notebook: train once on all 96 iSLEEPS patients,
then infer with no dataset-specific tuning — no fine-tuning, no threshold search, no
re-fitted normalisation, HMM transitions from iSLEEPS only.

**Two honest gaps.** ISRUC has no dedicated pulse channel and no separate "Effort" trace,
so 2 of the 7 cardiorespiratory inputs are zero-filled. Permutation importance put
pulse/HRV at 0.013 AUC, so this should cost a little respiratory performance and we should
expect it. And the cohort is 8 subjects recorded twice, not a large independent sample.

**Sessions are reported separately.** Respiratory-event prevalence is 10.1% in session 1
and 2.4% in session 2 — the second night is largely event-free for most subjects. Pooling
them would halve the apparent prevalence for a reason that has nothing to do with the
model, so session 1 is the primary result.

In [1]:
import glob
import json
import os
import sys
import time

import numpy as np
from sklearn.metrics import (accuracy_score, average_precision_score,
                             cohen_kappa_score, confusion_matrix, f1_score,
                             roc_auc_score)

REPO = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..", ".."))
sys.path.insert(0, os.path.join(REPO, "MMNet_research", "model"))
import mmnet_core as C  # noqa: E402

EXT = os.path.join(REPO, "data", "isruc_mm")
OUT = os.path.join(REPO, "MMNet_research", "results", "revision", "runs")
os.makedirs(OUT, exist_ok=True)
print("device:", C.DEV, "| iSLEEPS patients:", len(C.SUBS))
print("ISRUC recordings:", len(glob.glob(os.path.join(EXT, "*.npz"))))

cwd: D:\sleep-staging-psg\MMNet_research\MMNet_Submission\all_codes\notebooks | device: cuda | NVIDIA GeForce RTX 2060


subjects: 96 (SN28 dropped) | epochs: 89,532
stage %: {'W': np.float64(26.6), 'N1': np.float64(10.2), 'N2': np.float64(42.3), 'N3': np.float64(8.9), 'R': np.float64(12.1)}
respiratory-event prevalence: 16.0%
EEG-feature counts -> EEG: 112 EOG: 50 EMG: 26
parameters (concat): 773,254
training utilities defined.
device: cuda | iSLEEPS patients: 96
ISRUC recordings: 16


In [2]:
EXTDATA = {}
for f in sorted(glob.glob(os.path.join(EXT, "*.npz"))):
    tag = os.path.basename(f)[:-4]
    d = np.load(f)
    Fe = np.nan_to_num(d["Feeg"]).astype(np.float32)
    Fc = np.nan_to_num(d["Fcard"]).astype(np.float32)
    Fe = (Fe - Fe.mean(0)) / (Fe.std(0) + 1e-6)          # same as the iSLEEPS loader
    Fc = (Fc - Fc.mean(0)) / (Fc.std(0) + 1e-6)
    EXTDATA[tag] = (Fe, Fc, d["y"].astype(np.int64), d["apnea"].astype(np.int64),
                    int(d["session"]))

for sess in (1, 2):
    sel = [v for v in EXTDATA.values() if v[4] == sess]
    n = sum(len(v[2]) for v in sel)
    a = np.concatenate([v[3] for v in sel])
    print("session %d: %2d recordings | %5d epochs | respiratory prevalence %.1f%%"
          % (sess, len(sel), n, 100 * a.mean()))

yall = np.concatenate([v[2] for v in EXTDATA.values()])
print("\nISRUC stage %%:", {c: round(100 * (yall == i).mean(), 1) for i, c in enumerate(C.CLS)})
iy = np.concatenate([C.DATA[s][2] for s in C.SUBS])
print("iSLEEPS   %%:", {c: round(100 * (iy == i).mean(), 1) for i, c in enumerate(C.CLS)})

session 1:  8 recordings |  7122 epochs | respiratory prevalence 10.1%
session 2:  8 recordings |  7019 epochs | respiratory prevalence 2.4%

ISRUC stage %%: {'W': np.float64(15.7), 'N1': np.float64(15.6), 'N2': np.float64(35.7), 'N3': np.float64(18.4), 'R': np.float64(14.6)}
iSLEEPS   %%: {'W': np.float64(26.6), 'N1': np.float64(10.2), 'N2': np.float64(42.3), 'N3': np.float64(8.9), 'R': np.float64(12.1)}


## Train once on iSLEEPS, then infer

In [3]:
t0 = time.time()
rng = np.random.RandomState(42)
subs = list(C.SUBS); rng.shuffle(subs)
nv = max(10, len(subs) // 9)
va, tr = subs[:nv], subs[nv:]
model = C.train_fold(tr, va, "concat", [], [], seed=42)

Am = np.ones((C.NC, C.NC)); pi = np.ones(C.NC)
for s in tr:
    y = C.DATA[s][2]; pi[y[0]] += 1
    for a, b in zip(y[:-1], y[1:]): Am[a, b] += 1
A_log = np.log(Am / Am.sum(1, keepdims=True)); pi_log = np.log(pi / pi.sum())
print("trained on %d patients in %.1f min" % (len(tr), (time.time() - t0) / 60))

trained on 86 patients in 0.5 min


In [4]:
res = {}
for tag, (Fe, Fc, y, apn, sess) in EXTDATA.items():
    sp, ap = C.infer_arrays(model, Fe, Fc, len(y))
    pred = C.hmm(A_log, pi_log, np.log(sp + C.EPS))
    res[tag] = dict(sess=sess, y=y, pred=pred, apn=apn, score=ap)

def report(sessions, label):
    sel = [r for r in res.values() if r["sess"] in sessions]
    Y = np.concatenate([r["y"] for r in sel]); P = np.concatenate([r["pred"] for r in sel])
    A = np.concatenate([r["apn"] for r in sel]); S = np.concatenate([r["score"] for r in sel])
    out = dict(n_rec=len(sel), n_ep=int(len(Y)), prevalence=float(A.mean()),
               acc=float(accuracy_score(Y, P)),
               mf1=float(f1_score(Y, P, average="macro", zero_division=0)),
               kappa=float(cohen_kappa_score(Y, P)))
    if len(np.unique(A)) > 1:
        out["auc"] = float(roc_auc_score(A, S))
        out["ap"] = float(average_precision_score(A, S))
    print("%-22s rec %2d  ep %5d   acc %.4f  mF1 %.4f  kappa %.4f   AUC %s  AP %s"
          % (label, out["n_rec"], out["n_ep"], out["acc"], out["mf1"], out["kappa"],
             ("%.4f" % out["auc"]) if "auc" in out else "  n/a",
             ("%.4f" % out["ap"]) if "ap" in out else "  n/a"))
    return out

print("%-22s %s" % ("", "(iSLEEPS in-domain: acc 0.7227  mF1 0.6510  kappa 0.6106  AUC 0.7111  AP 0.3367)"))
print("-" * 108)
summary = {"session1": report([1], "ISRUC session 1"),
           "session2": report([2], "ISRUC session 2"),
           "both": report([1, 2], "ISRUC both sessions")}

                       (iSLEEPS in-domain: acc 0.7227  mF1 0.6510  kappa 0.6106  AUC 0.7111  AP 0.3367)
------------------------------------------------------------------------------------------------------------
ISRUC session 1        rec  8  ep  7122   acc 0.6641  mF1 0.6286  kappa 0.5570   AUC 0.7208  AP 0.2548
ISRUC session 2        rec  8  ep  7019   acc 0.5972  mF1 0.5507  kappa 0.4542   AUC 0.7198  AP 0.0487
ISRUC both sessions    rec 16  ep 14141   acc 0.6309  mF1 0.5906  kappa 0.5063   AUC 0.6976  AP 0.1435


## Per-stage transfer, and per-recording spread

In [5]:
sel = [r for r in res.values() if r["sess"] == 1]
Y = np.concatenate([r["y"] for r in sel]); P = np.concatenate([r["pred"] for r in sel])
cm = confusion_matrix(Y, P, labels=range(5))
rec_ = cm.diagonal() / np.maximum(cm.sum(1), 1)
f1s = f1_score(Y, P, average=None, labels=range(5), zero_division=0)

print("session 1, per stage")
print("%-5s %8s %10s %10s   %s" % ("stage", "n", "recall", "F1", "Sleep-EDF recall"))
sedf = [0.925, 0.247, 0.943, 0.554, 0.781]
for i, c in enumerate(C.CLS):
    print("%-5s %8d %10.3f %10.3f   %14.3f" % (c, cm.sum(1)[i], rec_[i], f1s[i], sedf[i]))

print("\nper-recording (session 1):")
for tag in sorted(t for t in res if res[t]["sess"] == 1):
    r = res[tag]
    a = ("%.3f" % roc_auc_score(r["apn"], r["score"])) if len(np.unique(r["apn"])) > 1 else " n/a"
    print("  %-7s n=%4d  acc %.3f  kappa %.3f  respAUC %s  prev %.1f%%"
          % (tag, len(r["y"]), accuracy_score(r["y"], r["pred"]),
             cohen_kappa_score(r["y"], r["pred"]), a, 100 * r["apn"].mean()))

accs = [accuracy_score(r["y"], r["pred"]) for r in sel]
print("  mean %.4f +- %.4f across recordings" % (np.mean(accs), np.std(accs)))

summary["per_stage_recall_s1"] = rec_.tolist()
summary["confusion_s1"] = cm.tolist()
json.dump(summary, open(os.path.join(OUT, "external_validation_isruc.json"), "w"), indent=1)
print("\nwrote external_validation_isruc.json")

session 1, per stage
stage        n     recall         F1   Sleep-EDF recall
W         1016      0.731      0.678            0.925
N1        1192      0.201      0.291            0.247
N2        2290      0.876      0.708            0.943
N3        1497      0.654      0.748            0.554
R         1127      0.676      0.718            0.781

per-recording (session 1):
  S1_1    n= 933  acc 0.458  kappa 0.323  respAUC 0.681  prev 8.5%
  S2_1    n= 851  acc 0.682  kappa 0.549  respAUC 0.848  prev 22.4%
  S3_1    n= 871  acc 0.727  kappa 0.617  respAUC 0.807  prev 7.1%
  S4_1    n= 932  acc 0.682  kappa 0.546  respAUC 0.825  prev 16.3%
  S5_1    n= 814  acc 0.748  kappa 0.654  respAUC 0.515  prev 1.4%
  S6_1    n= 965  acc 0.675  kappa 0.547  respAUC 0.649  prev 6.4%
  S7_1    n= 941  acc 0.629  kappa 0.532  respAUC 0.525  prev 13.4%
  S8_1    n= 815  acc 0.739  kappa 0.635  respAUC 0.682  prev 4.5%
  mean 0.6674 +- 0.0874 across recordings

wrote external_validation_isruc.json


## Reading the result

Two things are being tested at once and they should be read separately. The **staging**
number is the cleaner of the two: the EEG derivations match iSLEEPS exactly, so unlike
Sleep-EDF there is no montage mismatch to absorb the blame for any drop.

The **respiratory** number is the first external test of that head, and it carries a known
handicap: two of the seven cardiorespiratory inputs are zero-filled because ISRUC has no
pulse channel and no separate effort trace. A drop relative to the in-domain 0.711 is
therefore expected, and the honest question is whether detection survives at all rather
than whether it matches.